In [4]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from dotenv import load_dotenv

## 1. prepare paths and configurations

In [5]:
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
project_root

WindowsPath('h:/LLM/AbhishekCX/4_vector_stores')

In [6]:
pdf_path = project_root / "knowledge_base" / "beyond-chatbots-ai-agents-next-real-shift.pdf"
persist_directory = project_root / "pdf_db"
collection_name = "beyond-chatbots"
print(f"pdf_path :{pdf_path}")
print(f"persist_directory :{persist_directory}")
print(f"collection_name :{collection_name}")

pdf_path :h:\LLM\AbhishekCX\4_vector_stores\knowledge_base\beyond-chatbots-ai-agents-next-real-shift.pdf
persist_directory :h:\LLM\AbhishekCX\4_vector_stores\pdf_db
collection_name :beyond-chatbots


## Embeddings

In [7]:
embeddings = OllamaEmbeddings(model="qwen3-embedding:latest")
print("embeddings created")

embeddings created


## Pdf loaders

In [15]:
loader = PyPDFLoader(str(pdf_path))

In [16]:
docs = loader.load()
print(f"Total number of documents loaded: {len(docs)}")

Total number of documents loaded: 6


In [19]:
docs[0].page_content[:500]
docs[0].metadata

{'producer': 'ReportLab PDF Library - (opensource)',
 'creator': '(unspecified)',
 'creationdate': '2026-03-12T20:36:07+05:00',
 'author': 'By Editorial Desk',
 'keywords': '',
 'moddate': '2026-03-12T20:36:07+05:00',
 'subject': '(unspecified)',
 'title': 'Beyond Chatbots: Why AI Agents Feel Like the Next Real Shift',
 'trapped': '/False',
 'source': 'h:\\LLM\\AbhishekCX\\4_vector_stores\\knowledge_base\\beyond-chatbots-ai-agents-next-real-shift.pdf',
 'total_pages': 6,
 'page': 0,
 'page_label': '1'}

## Split the pdf into chunks

In [31]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)

In [32]:
chunked_docs = text_splitter.split_documents(docs)
print(f"Total number of chunked documents: {len(chunked_docs)}")

Total number of chunked documents: 88


In [34]:
print(chunked_docs[1].page_content)

The moment AI stopped feeling like a demo
For a long time, the most common experience with AI felt theatrical. You typed a question, the model
answered in polished language, and for a moment it seemed almost magical. Then the illusion broke. Ask a


## Store chunks in db

In [35]:
collection_name

'beyond-chatbots'

In [36]:
persist_directory

WindowsPath('h:/LLM/AbhishekCX/4_vector_stores/pdf_db')

In [37]:
vectorstore = Chroma.from_documents(
    documents=chunked_docs,
    embedding=embeddings,
    collection_name=collection_name,
    persist_directory=str(persist_directory)
)
print("Vector store created and documents added.")

Vector store created and documents added.


In [38]:
query = "How do AI agents use tools and memory?"
query

'How do AI agents use tools and memory?'

In [40]:
result = vectorstore.similarity_search(query, k=3)
result[0].page_content

'Page 1\n Beyond Chatbots: Why AI Agents Feel Like the\n Next Real Shift\nA practical long-form blog on planning, memory, tools, and retrieval in modern AI systems\nBy Editorial Desk\nThe moment AI stopped feeling like a demo'

In [42]:
result_with_scores = vectorstore.similarity_search_with_score(query, k=3)
result_with_scores

[(Document(id='ff136ff9-adeb-43ff-a0a8-d386c240143a', metadata={'producer': 'ReportLab PDF Library - (opensource)', 'moddate': '2026-03-12T20:36:07+05:00', 'subject': '(unspecified)', 'creator': '(unspecified)', 'author': 'By Editorial Desk', 'keywords': '', 'total_pages': 6, 'page': 0, 'page_label': '1', 'source': 'h:\\LLM\\AbhishekCX\\4_vector_stores\\knowledge_base\\beyond-chatbots-ai-agents-next-real-shift.pdf', 'trapped': '/False', 'creationdate': '2026-03-12T20:36:07+05:00', 'title': 'Beyond Chatbots: Why AI Agents Feel Like the Next Real Shift'}, page_content='Page 1\n Beyond Chatbots: Why AI Agents Feel Like the\n Next Real Shift\nA practical long-form blog on planning, memory, tools, and retrieval in modern AI systems\nBy Editorial Desk\nThe moment AI stopped feeling like a demo'),
  0.4057259261608124),
 (Document(id='eab9f36d-7150-4b04-9582-223a1acf039c', metadata={'page': 2, 'total_pages': 6, 'trapped': '/False', 'producer': 'ReportLab PDF Library - (opensource)', 'title': 